[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab02_measurement.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 2 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 2 — Measurement: rates, denominators, risk adjustment, suppression
*Week 2. **Unplugged**: no AI assistant this week. Baseline: sections 1–4. Powerup: section 5.*

The Hospital Readmissions Reduction Program (HRRP) file gives, for each hospital and condition, the
number of discharges, the number of readmissions, a *predicted* and an *expected* readmission rate,
and an *excess readmission ratio*. This week you reproduce the published numbers by hand and find out
exactly what each one means.

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


## 1. Load and read the file
Look at ten rows before computing anything. Note the values that are not numbers.

In [ ]:
hrrp = load_cms("hrrp")
print(hrrp.shape)
hrrp.head(10)

## 2. Suppression: the rows you cannot use
**Baseline:** for each `Measure Name`, count how many rows have `Number of Discharges` = `N/A`, how
many have `Number of Readmissions` = `Too Few to Report`, and how many are fully numeric. Then answer
in one sentence: why does CMS suppress small counts?

In [ ]:
# TODO: tabulate the three cases per measure. Hint: build boolean columns first.

**Your answer:** *(edit this cell)*

## 3. The raw rate, by hand
**Baseline:** on the usable rows only, compute `raw_rate = Number of Readmissions / Number of Discharges`
(as a percent). Compare it with `Predicted Readmission Rate` for the same rows:
1. scatter one against the other for one condition (heart failure, `READM-30-HF-HRRP`);
2. compute the correlation;
3. answer: why are they not equal, and which one would you show a patient?

In [ ]:
usable = hrrp[~hrrp["Number of Discharges"].eq("N/A") & ~hrrp["Number of Readmissions"].eq("Too Few to Report")].copy()
for c in ["Number of Discharges", "Number of Readmissions", "Excess Readmission Ratio",
          "Predicted Readmission Rate", "Expected Readmission Rate"]:
    usable[c] = to_num(usable[c])
# TODO: raw_rate, scatter vs predicted for READM-30-HF-HRRP, correlation.

**Your answer:** *(edit this cell)*

## 4. Reproduce the excess readmission ratio
CMS defines **Excess Readmission Ratio = Predicted / Expected**. *Expected* is what the model says an
average hospital would have with this hospital's patients.

**Baseline:** compute the ratio yourself and check it against the published column. Report the
maximum absolute difference and how many rows differ by more than 0.001. Then count hospitals with
ratio > 1 (penalized range) per measure.

In [ ]:
# TODO: my_err = predicted / expected; compare to the published column; count > 1 per measure.

## 5. Powerup — the risk-standardized rate
The number CMS publishes on Care Compare for the public is the **risk-standardized readmission rate**
= Excess Readmission Ratio × the national observed rate for that condition. The national rate is not in
this file.

**Powerup:** load `unplanned_visits` (`READM_30_*` measures carry the published risk-standardized
`Score`). For heart failure, join it to the HRRP rows by `Facility ID`, and estimate the national rate
as `Score / Excess Readmission Ratio`. If both numbers came from the same model run it would be the same
constant for every hospital. Report its median and spread. If it is *not* constant, look at the
`Start Date` / `End Date` columns of the two files before you blame the arithmetic.

In [ ]:
# TODO (powerup)
uhv = load_cms("unplanned_visits")
display(uhv["Measure ID"].value_counts())

## Milestone — question brief (due with this lab)
One page, in the cell below. Which track (quality / place and equity / cost), which question, why it
matters, which datasets. You will carry this question to week 14.

**Track:**

**Question (one sentence):**

**Why it matters (three sentences):**

**Data I will use:**

**What would change my mind:**